# Fase 3 — Umbral, A/B offline y análisis financiero
Este experimento replica el split y los modelos de Fase 2. Usa predicciones **out-of-fold del conjunto de desarrollo** para seleccionar umbral y modelo. La prueba final permanece reservada hasta el último paso.
Los costos son **supuestos hipotéticos**. McNemar contrasta desacuerdos en acierto/error entre dos modelos sobre las mismas observaciones; no es un A/B aleatorizado con clientes.


In [ ]:
%pip -q install ucimlrepo pandas numpy scikit-learn scipy matplotlib


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

SEED = 42
OUT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "fase3"
OUT.mkdir(parents=True, exist_ok=True)
data = fetch_ucirepo(id=350)
X = data.data.features.copy().drop(columns=["ID"], errors="ignore")
y = data.data.targets.iloc[:, 0].astype(int)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
cats = [c for c in ["SEX", "EDUCATION", "MARRIAGE"] if c in X.columns]
nums = [c for c in X.columns if c not in cats]
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), nums),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), cats)
])
models = {
    "Regresion_Logistica": LogisticRegression(max_iter=1500, random_state=SEED),
    "Random_Forest": RandomForestClassifier(n_estimators=120, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# Valores monetarios HIPOTÉTICOS por error; no representan pérdidas reales de un banco.
costs = {"FN": 5000, "FP": 300}
thresholds = np.round(np.arange(.10, .91, .05), 2)
oof, table = {}, []
for name, estimator in models.items():
    pipe = Pipeline([("prep", prep), ("model", estimator)])
    scores = cross_val_predict(pipe, X_dev, y_dev, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    oof[name] = scores
    for threshold in thresholds:
        pred = (scores >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_dev, pred, labels=[0,1]).ravel()
        table.append({"modelo":name, "umbral":threshold, "TN":tn, "FP":fp, "FN":fn, "TP":tp,
                      "precision":precision_score(y_dev,pred,zero_division=0),
                      "recall":recall_score(y_dev,pred,zero_division=0),
                      "f1":f1_score(y_dev,pred,zero_division=0),
                      "costo_hipotetico":int(fn*costs["FN"]+fp*costs["FP"])})
sweep = pd.DataFrame(table)
sweep.to_csv(OUT/"umbrales_desarrollo.csv",index=False)
# Selección EXCLUSIVAMENTE con desarrollo: minimizar costo hipotético; desempate por mayor F1.
selected = (sweep.sort_values(["costo_hipotetico","f1"],ascending=[True,False]).iloc[0])
chosen_name, chosen_t = str(selected["modelo"]), float(selected["umbral"])
print("Modelo/umbral seleccionado en desarrollo:",chosen_name,chosen_t)
print("Costos hipotéticos por error:",costs)
# A/B offline pareado, basado en predicciones OOF a umbral fijo 0.5: sin escoger umbral usando test.
pred_a = (oof["Regresion_Logistica"] >= .5).astype(int)
pred_b = (oof["Random_Forest"] >= .5).astype(int)
truth = y_dev.to_numpy()
a_ok, b_ok = pred_a == truth, pred_b == truth
a_only = int(np.sum(a_ok & ~b_ok))
b_only = int(np.sum(~a_ok & b_ok))
discordant = a_only+b_only
pvalue = float(binomtest(min(a_only,b_only),discordant,.5,alternative="two-sided").pvalue) if discordant else 1.0
ab = pd.DataFrame([{"A":"Regresion_Logistica","B":"Random_Forest","A_acierta_B_falla":a_only,
                    "A_falla_B_acierta":b_only,"discordancias":discordant,"mcnemar_exacto_p":pvalue,
                    "alpha":.05,"significativo":pvalue<.05}])
ab.to_csv(OUT/"comparacion_ab_mcnemar.csv",index=False)
print(ab.to_string(index=False))
# Sensibilidad de costos en desarrollo: costo de FN fijo; FP variable.
sens = []
for fp_cost in [100,300,600,1000,2000]:
    tmp=sweep.assign(costo=sweep.FN*costs["FN"]+sweep.FP*fp_cost)
    best=tmp.sort_values(["costo","f1"],ascending=[True,False]).iloc[0]
    sens.append({"costo_FP":fp_cost,"costo_FN":costs["FN"],"modelo":best.modelo,
                 "umbral":best.umbral,"costo_total":best.costo})
pd.DataFrame(sens).to_csv(OUT/"sensibilidad_costos.csv",index=False)
# Evaluación final una sola vez: modelo y umbral fijados con desarrollo.
final = Pipeline([("prep",prep),("model",models[chosen_name])])
final.fit(X_dev,y_dev)
test_score = final.predict_proba(X_test)[:,1]
test_pred = (test_score >= chosen_t).astype(int)
tn,fp,fn,tp=confusion_matrix(y_test,test_pred,labels=[0,1]).ravel()
metrics={"modelo":chosen_name,"umbral":chosen_t,"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp),
         "accuracy":accuracy_score(y_test,test_pred),"precision":precision_score(y_test,test_pred,zero_division=0),
         "recall":recall_score(y_test,test_pred,zero_division=0),"f1":f1_score(y_test,test_pred,zero_division=0),
         "roc_auc":roc_auc_score(y_test,test_score),
         "costo_hipotetico":int(fn*costs["FN"]+fp*costs["FP"])}
pd.DataFrame([metrics]).to_csv(OUT/"evaluacion_final_test.csv",index=False)
print("Evaluación final (test reservado):",metrics)
fig,ax=plt.subplots(figsize=(8,5))
for name,group in sweep.groupby("modelo"):
    ax.plot(group.umbral,group.costo_hipotetico,label=name,marker=".")
ax.axvline(chosen_t,ls="--",color="gray",label=f"Umbral seleccionado {chosen_t:.2f}")
ax.set(xlabel="Umbral",ylabel="Costo hipotético (unidades monetarias)",title="Costos por umbral (predicciones OOF, desarrollo)")
ax.legend();fig.tight_layout();fig.savefig(OUT/"costos_por_umbral.png",dpi=150);plt.close(fig)
print("Archivos generados:",sorted(p.name for p in OUT.iterdir()))


## Interpretación
- El umbral se elige por **costo hipotético mínimo** en predicciones fuera de fold, no por el conjunto de prueba.
- McNemar exacto usa una prueba binomial sobre desacuerdos pareados; su p-valor no mide tamaño de impacto económico ni causalidad.
- El análisis de sensibilidad cambia el costo de falsos positivos para evaluar robustez de la elección.
- El conjunto de prueba solo se evalúa después de seleccionar modelo y umbral; los resultados se registran en `results/fase3/`.
- Los datos históricos no permiten validar despliegue productivo ni estimar costos reales.
